# Chapter 9.6: three SKU columns from five inputs

Five inputs: beer to bright tank a hL, overhead a hL, freight a hL, the declared price a hL and the pack table.
Each SKU must tie to its ex works cost a hL. Excise stays outside the cost. Every figure is synthetic.

In [ ]:
import sys
sys.path.append('..')
import pandas as pd
import matplotlib.pyplot as plt
from calculator import inr, r2

In [ ]:
BEER_HL, OVERHEAD_HL, FREIGHT_HL, DMC_HL = 2880, 1890, 630, 12600
packs = pd.read_csv('../data/skus.csv').set_index('sku_id').loc[['SKU1', 'SKU2', 'SKU3']]

def columns(beer_hl=BEER_HL, overhead_hl=OVERHEAD_HL, freight_hl=FREIGHT_HL, dmc_hl=DMC_HL, packs=packs):
    t = pd.DataFrame(index=packs.index)
    t['hl_case'] = packs.hl_case
    t['beer'] = (packs.hl_case * beer_hl).round(2)
    t['packaging'] = (packs.units * packs.unit_cost + packs.carton_cost).round(2)
    t['overhead'] = (packs.hl_case * overhead_hl).round(2)
    t['ex_works'] = t.beer + t.packaging + t.overhead
    t['ex_works_hl'] = (t.ex_works / t.hl_case).round(2)
    t['freight'] = (packs.hl_case * freight_hl).round(2)
    t['delivered'] = t.ex_works + t.freight
    t['dmc'] = (packs.hl_case * dmc_hl).round(2)
    t['contribution'] = (t.dmc - t.beer - t.packaging - t.freight).round(2)
    t['margin'] = (t.dmc - t.delivered).round(2)
    money = t.columns.drop('hl_case')
    t[money] = t[money].round(2)
    return t

cpc = columns()
# Each SKU ties to its ex works a hL: the case total over hL a case must equal
# beer a hL plus overhead a hL plus packaging a hL (paisa rounding of the lines allowed).
for sku, r in cpc.iterrows():
    built_up = BEER_HL + OVERHEAD_HL + r.packaging / r.hl_case
    assert abs(r.ex_works_hl - built_up) < 0.15, sku
cpc.T

In [ ]:
pack_share = (cpc.packaging / cpc.ex_works * 100).round(1)
usd_hl = (cpc.ex_works_hl / 90).round(2)
gap_sku1 = r2(cpc.ex_works_hl.SKU1 - (2880 + 2790 + 1890))
print('packaging share of ex works:', dict(pack_share))
print('ex works a hL in USD:', dict(usd_hl))
print('SKU 1 above the blended 7,560 a hL by', gap_sku1, '; packaging a hL', r2(246 / 0.078), 'against 2,790')
print('can against bottle: delivered +', r2(cpc.delivered.SKU3 - cpc.delivered.SKU1), '; margin -', r2(cpc.margin.SKU1 - cpc.margin.SKU3),
      '; DMC +', r2(cpc.dmc.SKU3 - cpc.dmc.SKU1))
print('DMC read as cost overstates SKU 1 cost by', r2(cpc.dmc.SKU1 - cpc.ex_works.SKU1))

## Malt up 20 percent: which line moves?

In [ ]:
malt_hl = 5300 * 81 / 300
stress_hl = 0.20 * malt_hl
stressed = columns(beer_hl=BEER_HL + stress_hl)
print('malt a hL', malt_hl, '; stress adds', r2(stress_hl), 'a hL,', r2(stress_hl * 0.078), 'a 650 ml case')
(stressed[['beer', 'packaging', 'ex_works']] - cpc[['beer', 'packaging', 'ex_works']]).round(2)